In [ ]:
!pip install -q fastapi uvicorn pyngrok "diffusers[torch]" accelerate safetensors

from fastapi import FastAPI
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video
import torch, tempfile, os, threading, uvicorn

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe_t2v = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16",
)
pipe_t2v.enable_model_cpu_offload()
pipe_t2v.enable_vae_slicing()

class Txt2VideoRequest(BaseModel):
    prompt: str
    num_frames: int = 16
    fps: int = 8

@app.post("/api/text2video")
def text2video(req: Txt2VideoRequest):
    result = pipe_t2v(
        req.prompt,
        num_frames=req.num_frames,
        num_inference_steps=25,
        guidance_scale=7.5,
    )
    frames = result.frames[0]
    tmp = tempfile.mkdtemp()
    video_path = os.path.join(tmp, "out.mp4")
    export_to_video(frames, video_path, fps=req.fps)
    return FileResponse(video_path, media_type="video/mp4")

def run():
    uvicorn.run(app, host="0.0.0.0", port=8001)

thread = threading.Thread(target=run, daemon=True)
thread.start()

ngrok.set_auth_token("37KKsS5VIS3ttwVmeFQk2apDnb5_3i9ZkGWYyxfeG6CfFakzf")
public_url_t2v = ngrok.connect(8001, "http")
public_url_t2v


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--damo-vilab--text-to-video-ms-1.7b/snapshots/8227dddca75a8561bf858d604cc5dae52b954d01/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 
/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `TextToVideoSDPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


INFO:     Started server process [346]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


<NgrokTunnel: "https://proart-nonfictively-georgetta.ngrok-free.dev" -> "http://localhost:8001">